# Run the Pathway Model with LCA metrics

## Run the model

In [ ]:
from shared.utils import run_pathway
import pandas as pd
import plotly.express as px

In [ ]:
lca_normalization_factors = pd.read_csv('../02_AMPL_files/data/QC_techs_lca_max.csv')

In [ ]:
lca_files = [
    '../02_AMPL_files/model/QC_objectives_lca.mod',
    '../02_AMPL_files/model/QC_objectives_lca_direct.mod',
    '../02_AMPL_files/model/QC_objectives_lca_territorial.mod',
    '../02_AMPL_files/data/QC_techs_lca.dat',
    '../02_AMPL_files/data/QC_techs_lca_direct.dat',
    '../02_AMPL_files/data/QC_techs_lca_territorial.dat',
]

In [ ]:
results = run_pathway(case_study='pathway_lca', extra_files=lca_files)

## Extract results

In [ ]:
# All available result keys
print([k for k in results if results[k] is not None])

In [ ]:
def plot_lca_indicator(results, indicator, metric):

    lca_results_phase = pd.merge(
        results[metric].reset_index(),
        lca_normalization_factors,
        left_on='Indicators',
        right_on='Abbrev',
        how='left',
    )

    # By default, metrics are in kg CO2-eq/kW(h) or DALY/tkm for instance. There is a 1e-6 factor to account for.
    lca_results_phase[f'{metric} (physical units x1e-6)'] = lca_results_phase[metric] * lca_results_phase['max_unit']
    lca_results_phase[f'{metric} (physical units x1e-9)'] = lca_results_phase[f'{metric} (physical units x1e-6)'] * 1e-3

    if indicator == 'm_CCS_all':
        unit = 'Mt CO2-eq'
        name = 'Climate change'
        data = f'{metric} (physical units x1e-9)'
    elif indicator == 'RHHD':
        unit = 'x1e-6 DALY'
        name = 'Remaining human health damage'
        data = f'{metric} (physical units x1e-6)'
    elif indicator == 'REQD':
        unit = 'x1e-9 PDF.m2.yr'
        name = 'Remaining ecosystem quality damage'
        data = f'{metric} (physical units x1e-9)'
    elif indicator == 'TTHH':
        unit = 'x1e-6 DALY'
        name = 'Total human health damage'
        data = f'{metric} (physical units x1e-6)'
    elif indicator == 'TTEQ':
        unit = 'x1e-9 PDF.m2.yr'
        name = 'Total ecosystem quality damage'
        data = f'{metric} (physical units x1e-9)'
    else:
        raise ValueError(f'Unrecognized indicator: {indicator}')

    fig = px.line(
        lca_results_phase[(lca_results_phase['Indicators'] == indicator) & (lca_results_phase['Phases'] != '2015_2020')].groupby(['Phases'])[data].sum().reset_index(),
        x='Phases',
        y=data,
        labels={data: f'{name} ({unit})'},
    )

    fig.show()

In [ ]:
plot_lca_indicator(results, 'm_CCS_all', 'PhaseLCIA')

In [ ]:
plot_lca_indicator(results, 'REQD', 'LCIA_constr')

In [ ]:
plot_lca_indicator(results, 'RHHD', 'LCIA_op')